# Opdracht 1: Voorspel herseninfarct (MedPredict)

**Team:** *AMK-KLAS B*

**Teamleden en Kaggle-gebruikersnamen:**

| Naam | Kaggle-gebruikersnaam |
|---|---|
| **Ahmed Bashabeeb 25097172** | ahmedbashabeeb |
| **Khaleel Al-Aqel 24053872** | ** | 
| **Mehmet Kilinc 25135007** | *khaleelalaqel* |



## 1. Inleiding

In deze opdracht ontwikkelen we voor MedPredict een machine learning-model dat voorspelt of een patiënt een herseninfarct (stroke) krijgt, op basis van gegevens. We werken hiervoor met een dataset van ruim 33.000 patiënten en leveren onze voorspellingen in via een Kaggle-competitie. Naast een zo goed mogelijk voorspellend model kijken we ook naar de medische context: een gemiste


## 2. Data

We gebruiken drie bestanden uit de Kaggle-competitie
["Predicting stroke 2026-27"](https://www.kaggle.com/competitions/predicting-stroke-2026-27):

- **train.csv** — 33.550 patiënten, inclusief doelvariabele `stroke` (0/1).
- **test.csv** — 8.388 patiënten, zonder `stroke`-kolom.
- **submission_example.csv** — voorbeeldformaat voor de Kaggle-inlevering (`id`, `stroke`).


In [1]:
# Prompt 1: <CHAT-TITEL>
"""Imports en algemene instellingen voor het hele notebook."""
import time
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from imblearn.pipeline import Pipeline
from imblearn.under_sampling import RandomUnderSampler
from IPython.display import display
from sklearn.base import clone
from sklearn.ensemble import (GradientBoostingClassifier,
                              RandomForestClassifier, VotingClassifier)
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (ConfusionMatrixDisplay, accuracy_score,
                             average_precision_score, confusion_matrix,
                             f1_score, make_scorer, precision_recall_curve,
                             precision_score, recall_score, roc_auc_score)
from sklearn.model_selection import (GridSearchCV, StratifiedKFold,
                                     cross_val_predict, train_test_split)
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier

# Vaste instellingen zodat alle resultaten reproduceerbaar zijn
RANDOM_STATE = 42
N_FOLDS = 5
N_JOBS = -1  # gebruik alle CPU-kernen
DATA_DIR = Path("data")
SUBMISSION_DIR = Path("submissions")
SUBMISSION_DIR.mkdir(exist_ok=True)

# Waarschuwingen (bijv. over convergentie tijdens het tunen) maken de
# output onoverzichtelijk; daarom zetten we ze uit.
warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.dpi"] = 100
pd.set_option("display.max_columns", 30)
pd.set_option("display.precision", 3)


<br>

### 2.2 Klasse `StrokeDataset`: data laden, controleren en opschonen
Deze klasse bundelt alles wat met de ruwe data te maken heeft. Door dit in één klasse te zetten, worden train- en testdata altijd op precies dezelfde manier bewerkt. Dat voorkomt fouten zoals een kolom die wel uit train maar niet uit test wordt verwijderd.

| Methode | Wat doet het? |
|---|---|
| `load()` | Leest `train.csv` en `test.csv` in. |
| `column_overview()` | Datatype, aantal unieke en ontbrekende waarden, minimum en maximum per kolom. |
| `check_sklearn_requirements()` | Controleert de minimale eisen van scikit-learn (paragraaf 3.5). |
| `readable_frame()` | Zet de dummy-kolommen terug naar leesbare categorieën, alleen voor de grafieken. |
| `clean()` | Schoont train en test op dezelfde manier op (paragraaf 3.7). |
| `split()` | Maakt een gestratificeerde train- en validatieset. |

In [2]:
# Prompt 1: <CHAT-TITEL>
class StrokeDataset:
    """Laadt, controleert en schoont de stroke-data van Kaggle.

    Parameters
    ----------
    data_dir : Path
        Map waarin train.csv en test.csv staan.
    """

    TARGET = "stroke"
    ID_COLUMN = "id"
    NUMERIC_COLUMNS = ["age", "avg_glucose_level", "bmi"]
    # Van iedere binaire variabele houden we één dummy-kolom over
    REDUNDANT_COLUMNS = ["gender_Female", "ever_married_No",
                         "Residence_type_Rural"]
    DUMMY_GROUPS = ["gender", "ever_married", "work_type",
                    "Residence_type", "smoking_status"]

    def __init__(self, data_dir):
        self.data_dir = Path(data_dir)
        self.train_raw = None
        self.test_raw = None
        self.train = None
        self.test = None
        self.test_ids = None
        self.cleaning_log = []

    def load(self):
        """Leest train.csv en test.csv in als DataFrames."""
        self.train_raw = pd.read_csv(self.data_dir / "train.csv")
        self.test_raw = pd.read_csv(self.data_dir / "test.csv")
        return self

    def column_overview(self):
        """Geeft per kolom datatype, uniek, ontbrekend, min en max."""
        data = self.train_raw
        return pd.DataFrame({
            "datatype": data.dtypes.astype(str),
            "aantal uniek": data.nunique(),
            "ontbrekend": data.isna().sum(),
            "minimum": data.min(),
            "maximum": data.max(),
        })

    def check_sklearn_requirements(self):
        """Controleert de minimale eisen van scikit-learn voor X en y."""
        train, test = self.train_raw, self.test_raw
        features = train.columns.drop(self.TARGET)
        checks = {
            "Alle kolommen zijn numeriek of boolean": all(
                pd.api.types.is_numeric_dtype(train[column])
                for column in train.columns),
            "Geen ontbrekende waarden in train":
                int(train.isna().sum().sum()) == 0,
            "Geen ontbrekende waarden in test":
                int(test.isna().sum().sum()) == 0,
            "Geen dubbele patiënten (rijen zonder id)":
                not train.drop(columns=self.ID_COLUMN).duplicated().any(),
            "Zelfde feature-kolommen in train en test":
                list(features) == list(test.columns),
            "Doelvariabele is binair (0/1)":
                set(train[self.TARGET].unique()) == {0, 1},
        }
        return pd.DataFrame({"voldaan": list(checks.values())},
                            index=list(checks.keys()))

    def readable_frame(self):
        """Zet de dummy-kolommen terug naar categorieën (voor de EDA)."""
        data = self.train_raw.copy()
        for group in self.DUMMY_GROUPS:
            prefix = group + "_"
            columns = [col for col in data.columns if col.startswith(prefix)]
            labels = data[columns].idxmax(axis=1).str.replace(
                prefix, "", regex=False)
            # Rijen waarin alle dummies False zijn: categorie 'Unknown'
            labels[~data[columns].any(axis=1)] = "Unknown"
            data[group] = labels
            data = data.drop(columns=columns)
        return data

    def clean(self):
        """Schoont train en test op (motivatie: paragraaf 3.7)."""
        train = self.train_raw.copy()
        test = self.test_raw.copy()

        # 1. Rijen met gender 'Other' zijn te zeldzaam om van te leren
        other_rows = train["gender_Other"]
        train = train.loc[~other_rows]
        self._log(f"{other_rows.sum()} rijen met gender 'Other' "
                  "verwijderd uit train")

        # 2. Id bewaren voor de submission en daarna als feature weghalen
        self.test_ids = test[self.ID_COLUMN].copy()
        drop_columns = ([self.ID_COLUMN, "gender_Other"]
                        + self.REDUNDANT_COLUMNS)
        train = train.drop(columns=drop_columns)
        test = test.drop(columns=drop_columns)
        self._log("kolommen verwijderd: " + ", ".join(drop_columns))

        # 3. Booleans omzetten naar 0/1 zodat alle features getallen zijn
        bool_columns = train.select_dtypes(include="bool").columns
        train[bool_columns] = train[bool_columns].astype(int)
        test[bool_columns] = test[bool_columns].astype(int)
        self._log(f"{len(bool_columns)} boolean-kolommen omgezet naar 0/1")

        self.train = train.reset_index(drop=True)
        self.test = test.reset_index(drop=True)
        return self

    def _log(self, message):
        """Bewaart een regel tekst over een opschoonstap."""
        self.cleaning_log.append(message)

    def get_features_and_target(self):
        """Geeft de features X en doelvariabele y van de opgeschoonde train."""
        features = self.train.drop(columns=self.TARGET)
        target = self.train[self.TARGET]
        return features, target

    def split(self, test_size=0.2):
        """Maakt een gestratificeerde train- en validatieset."""
        features, target = self.get_features_and_target()
        return train_test_split(features, target, test_size=test_size,
                                stratify=target,
                                random_state=RANDOM_STATE)

<br>

### 2.3 Klasse `EdaPlotter`: visualisaties voor de EDA
Alle grafieken van de EDA worden door één klasse gemaakt. Daardoor hebben ze dezelfde opmaak en is de code van hoofdstuk 3 kort.

In [4]:
# Prompt 1: <CHAT-TITEL>
class EdaPlotter:
    """Maakt de visualisaties voor de exploratieve data-analyse.

    Parameters
    ----------
    data : pd.DataFrame
        Leesbare dataset (categorieën in plaats van dummy-kolommen).
    target : str
        Naam van de doelvariabele.
    """

    COLORS = {0: "#4C72B0", 1: "#C44E52"}

    def __init__(self, data, target="stroke"):
        self.data = data
        self.target = target
        self.overall_rate = data[target].mean() * 100

    def target_distribution(self):
        """Toont het aantal en percentage patiënten per klasse."""
        counts = self.data[self.target].value_counts().sort_index()
        fig, ax = plt.subplots(figsize=(7, 4))
        bars = ax.bar(["Geen stroke (0)", "Stroke (1)"], counts.values,
                      color=list(self.COLORS.values()))
        for bar, count in zip(bars, counts.values):
            share = count / counts.sum() * 100
            ax.annotate(f"{count:,} ({share:.1f}%)",
                        (bar.get_x() + bar.get_width() / 2,
                         bar.get_height()),
                        ha="center", va="bottom")
        ax.set_title("Verdeling van de doelvariabele stroke")
        ax.set_ylabel("Aantal patiënten")
        plt.tight_layout()
        plt.show()

    def numeric_distributions(self, columns):
        """Toont per numerieke feature een dichtheidsplot en boxplot."""
        fig, axes = plt.subplots(2, len(columns),
                                 figsize=(5.5 * len(columns), 8))
        for i, column in enumerate(columns):
            sns.kdeplot(data=self.data, x=column, hue=self.target,
                        common_norm=False, fill=True,
                        palette=self.COLORS, ax=axes[0, i])
            axes[0, i].set_title(f"Dichtheid van {column} per klasse")
            sns.boxplot(data=self.data, x=self.target, y=column,
                        hue=self.target, palette=self.COLORS,
                        legend=False, ax=axes[1, i])
            axes[1, i].set_title(f"Boxplot van {column} per klasse")
        plt.tight_layout()
        plt.show()

    def rate_per_category(self, columns):
        """Toont het stroke-percentage per categorie van iedere feature."""
        n_cols = 4
        n_rows = int(np.ceil(len(columns) / n_cols))
        fig, axes = plt.subplots(n_rows, n_cols, figsize=(19, 4.6 * n_rows))
        axes = axes.flatten()
        for ax, column in zip(axes, columns):
            summary = self.data.groupby(column)[self.target].agg(
                ["mean", "size"])
            ax.bar(summary.index.astype(str), summary["mean"] * 100,
                   color="#55A868")
            ax.axhline(self.overall_rate, color="red", linestyle="--",
                       label=f"gemiddeld ({self.overall_rate:.2f}%)")
            for x_pos, (rate, size) in enumerate(
                    zip(summary["mean"], summary["size"])):
                ax.annotate(f"n={size:,}", (x_pos, rate * 100),
                            ha="center", va="bottom", fontsize=8)
            ax.set_title(f"Stroke-% per {column}")
            ax.set_ylabel("Stroke (%)")
            ax.tick_params(axis="x", rotation=30)
        for ax in axes[len(columns):]:
            ax.set_visible(False)
        axes[0].legend(loc="upper left")
        plt.tight_layout()
        plt.show()

    def rate_per_age_group(self, bin_width=5):
        """Toont het stroke-percentage per leeftijdsgroep."""
        bins = np.arange(0, self.data["age"].max() + bin_width, bin_width)
        groups = pd.cut(self.data["age"], bins=bins, right=False)
        summary = self.data.groupby(groups, observed=True)[
            self.target].agg(["mean", "size"])
        fig, ax = plt.subplots(figsize=(11, 4))
        ax.plot([interval.left for interval in summary.index],
                summary["mean"] * 100, marker="o", color="#C44E52")
        ax.set_title("Stroke-% per leeftijdsgroep van 5 jaar")
        ax.set_xlabel("Leeftijd (ondergrens van de groep)")
        ax.set_ylabel("Stroke (%)")
        plt.tight_layout()
        plt.show()

    def target_correlations(self, numeric_data):
        """Toont de correlatie van iedere feature met de target."""
        target_corr = (numeric_data.corr()[self.target]
                       .drop(self.target).sort_values())
        fig, ax = plt.subplots(figsize=(9, 6))
        ax.barh(target_corr.index, target_corr.values,
                color=np.where(target_corr > 0, "#C44E52", "#4C72B0"))
        ax.set(title="Correlatie (Pearson) van iedere feature met stroke",
               xlabel="Correlatie")
        plt.tight_layout()
        plt.show()

<br>

<br>

<br>

## 3. Opdracht 1 – Exploratieve Data Analyse (EDA)

### 3.1 Kaggle-team
Ieder teamlid heeft een Kaggle-account aangemaakt en we doen als één team mee aan de competitie [Predicting stroke 2026-27](https://www.kaggle.com/competitions/predicting-stroke-2026-27). De teamnaam, onze namen en Kaggle-gebruikersnamen staan bovenaan dit notebook.

### 3.2 Data inladen

In [6]:
# Prompt 1: <CHAT-TITEL>
dataset = StrokeDataset(DATA_DIR).load()
print(f"train.csv: {dataset.train_raw.shape[0]:,} rijen, "
      f"{dataset.train_raw.shape[1]} kolommen")
print(f"test.csv:  {dataset.test_raw.shape[0]:,} rijen, "
      f"{dataset.test_raw.shape[1]} kolommen (zonder 'stroke')")
dataset.train_raw.head()

FileNotFoundError: [Errno 2] No such file or directory: 'data\\train.csv'

### 3.3 Uitleg van iedere kolom
De dataset is al gedeeltelijk voorbewerkt: de categorische variabelen zijn met **one-hot encoding** omgezet naar dummy-kolommen. Iedere categorie heeft een eigen kolom met `True` of `False`.

| Kolom | Type | Betekenis |
|---|---|---|
| `id` | geheel getal | Uniek nummer van de patiënt. Bevat geen medische informatie. |
| `age` | decimaal getal | Leeftijd in jaren. Bij baby's is dit een breuk (bijv. 0,08 jaar ≈ 1 maand). |
| `hypertension` | 0/1 | 1 = de patiënt heeft hoge bloeddruk (hypertensie). |
| `heart_disease` | 0/1 | 1 = de patiënt heeft een hartziekte. |
| `avg_glucose_level` | decimaal getal | Gemiddelde bloedsuikerspiegel (glucose) in mg/dL. |
| `bmi` | decimaal getal | Body Mass Index: gewicht (kg) gedeeld door lengte (m) in het kwadraat. |
| `gender_Female` | boolean | Geslacht is vrouw. |
| `gender_Male` | boolean | Geslacht is man. |
| `gender_Other` | boolean | Geslacht is anders. |
| `ever_married_No` | boolean | De patiënt is nooit getrouwd geweest. |
| `ever_married_Yes` | boolean | De patiënt is ooit getrouwd (geweest). |
| `work_type_Govt_job` | boolean | Werkt bij de overheid. |
| `work_type_Never_worked` | boolean | Heeft nooit gewerkt. |
| `work_type_Private` | boolean | Werkt in het bedrijfsleven (private sector). |
| `work_type_Self-employed` | boolean | Is zelfstandig ondernemer. |
| `work_type_children` | boolean | Is een kind en werkt (nog) niet. |
| `Residence_type_Rural` | boolean | Woont op het platteland. |
| `Residence_type_Urban` | boolean | Woont in de stad. |
| `smoking_status_formerly smoked` | boolean | Heeft vroeger gerookt. |
| `smoking_status_never smoked` | boolean | Heeft nooit gerookt. |
| `smoking_status_smokes` | boolean | Rookt op dit moment. |
| `stroke` | 0/1 | **Doelvariabele**: 1 = de patiënt heeft een herseninfarct gehad. Ontbreekt in `test.csv`. |

**Let op bij `smoking_status`:** bij ongeveer 31% van de patiënten zijn alle drie de rookkolommen `False`. Dat is de categorie *Unknown* (rookstatus onbekend); die kolom is bij het maken van de dummies weggelaten. Bijna de helft van deze groep is jonger dan 18 jaar.

<br>

### 3.4 Datatypes en basisstatistieken

In [7]:
# Datatype, aantal unieke en ontbrekende waarden, minimum en maximum
dataset.column_overview()

NameError: name 'dataset' is not defined

In [ ]:
# Datatype, aantal unieke en ontbrekende waarden, minimum en maximum
dataset.column_overview()

**Interpretatie**
- `age` loopt van 0,08 tot 82 jaar (gemiddeld 41,8). Er zitten dus ook baby's en kinderen in de data.
- `avg_glucose_level` loopt van 55 tot 282 mg/dL. Het gemiddelde (103,6) ligt boven de mediaan (91,3): de verdeling is **rechtsscheef**, met een groep patiënten met een hoge bloedsuiker.
- `bmi` loopt van 10,1 tot 97,6. Waarden boven de 60 zijn extreem maar medisch mogelijk (ernstig overgewicht); zie paragraaf 3.7.
- 8,8% heeft hoge bloeddruk en 4,3% een hartziekte.
- Slechts **1,54%** van de patiënten heeft een herseninfarct gehad (`stroke` = 1). De dataset is dus sterk **imbalanced**.
- De features hebben heel verschillende schalen (0/1 tegenover 55–282). Dat is belangrijk voor standaardisatie (paragraaf 3.8).

### 3.5 Voldoet de data aan de minimale eisen van scikit-learn?
Scikit-learn stelt een aantal minimale eisen aan de data (Pedregosa et al., 2011; scikit-learn developers, z.d.-a):

1. **Alles is numeriek.** Modellen rekenen met getallen; tekst (zoals "Male") moet eerst gecodeerd worden.
2. **Geen ontbrekende waarden.** De meeste estimators geven een foutmelding bij `NaN`.
3. **De juiste vorm.** De features `X` zijn een 2D-tabel (één rij per patiënt, één kolom per feature) en de target `y` is een 1D-reeks met één label per rij.
4. **Dezelfde kolommen in train en test**, in dezelfde volgorde. Een model dat op 20 features is getraind, kan geen voorspelling doen met 19 features.
5. **Alleen zinvolle features.** Kolommen zonder voorspellende betekenis, zoals een id, horen niet in `X`.

In [ ]:
dataset.check_sklearn_requirements()

**Conclusie:** de data voldoet aan de technische eisen. Alle kolommen zijn numeriek of boolean (scikit-learn ziet `True`/`False` als 1/0), er ontbreekt niets, er zijn geen dubbele patiënten en train en test hebben dezelfde features.

Toch passen we een paar dingen aan (paragraaf 3.7):
- de kolom `id` verwijderen, omdat het geen kenmerk van de patiënt is (eis 5);
- de booleans expliciet omzetten naar 0/1, zodat alle features hetzelfde datatype hebben;
- overbodige dummy-kolommen verwijderen.



### 3.6 Relaties tussen de onafhankelijke variabelen en `stroke`
Voor de grafieken zetten we de dummy-kolommen tijdelijk terug naar één leesbare kolom per variabele, bijvoorbeeld `work_type` met de waarden *Private*, *Self-employed*, enzovoort.

#### 3.6.1 Verdeling van de doelvariabele

In [ ]:
eda_data = dataset.readable_frame()
plotter = EdaPlotter(eda_data)
plotter.target_distribution()

Van de 33.550 patiënten hebben er maar **517 (1,54%)** een herseninfarct gehad. Een model dat altijd "geen stroke" voorspelt, heeft daardoor al een accuracy van 98,5%, maar vindt geen enkele risicopatiënt. Daarom is accuracy hier een slechte maatstaf (hoofdstuk 4) en hebben we speciale technieken nodig (paragraaf 5.1).

#### 3.6.2 Numerieke features: leeftijd, glucose en BMI

In [ ]:
plotter.numeric_distributions(StrokeDataset.NUMERIC_COLUMNS)

**Interpretatie**
- **`age` is de sterkste voorspeller.** Patiënten met stroke zijn veel ouder: mediaan 72 jaar tegenover 43 jaar. De dichtheidsplots overlappen nauwelijks aan de linkerkant: onder de 30 jaar komt stroke bijna niet voor.
- **`avg_glucose_level`** heeft bij beide groepen een piek rond 80–100 mg/dL, maar bij stroke-patiënten is de **tweede piek** rond 200+ mg/dL (diabetesniveau) veel groter. Gemiddeld 129,5 tegenover 103,2.
- **`bmi`** verschilt nauwelijks (mediaan 29,0 tegenover 27,7). Op zichzelf voorspelt BMI dus weinig.

#### 3.6.3 Binaire en categorische features
Per categorie tonen we het percentage patiënten met stroke. De rode stippellijn is het gemiddelde (1,54%) en `n` is het aantal patiënten in die categorie.

In [1]:
categorical_columns = ["hypertension", "heart_disease", "gender",
                       "ever_married", "work_type", "Residence_type",
                       "smoking_status"]
plotter.rate_per_category(categorical_columns)

NameError: name 'plotter' is not defined